# F1TENTH Multi-Agent Training Analysis Dashboard

Este notebook contiene un análisis completo del rendimiento de las funciones de recompensa en el entrenamiento multi-agente de F1TENTH.

## Métricas Recolectadas
- **Progreso de vuelta** (`lap_progress`): Progreso normalizado en la pista
- **Tiempo de vuelta** (`lap_time`): Tiempo para completar una vuelta 
- **Velocidad promedio** (`avg_speed`): Velocidad promedio durante el episodio
- **Duración del episodio** (`episode_duration`): Tiempo total del episodio
- **Colisiones** (`total_collisions`): Número total de colisiones
- **Recompensa** (`episode_reward_mean`): Recompensa promedio por episodio

## Funciones de Recompensa Analizadas
1. **ProgressRewardAdvancedEnv**: Recompensa basada en progreso avanzado
2. **SpeedRewardEnv**: Recompensa enfocada en velocidad
3. **WaypointRewardEnv**: Recompensa basada en waypoints
4. **SafetyRewardEnv**: Recompensa enfocada en seguridad

## 📦 Imports y Configuración

In [ ]:
# Imports básicos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob
import json
from typing import Dict, List, Optional

# Ray/RLlib imports para análisis
import ray
from ray.tune.analysis import ExperimentAnalysis

# Configuración de plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Configuración de warnings
import warnings
warnings.filterwarnings('ignore')

# Configuración para mostrar todos los plots inline
%matplotlib inline

print("✅ Imports completados")

In [ ]:
# Funciones helper para cargar datos
def load_experiment_data(experiment_paths: List[str]) -> pd.DataFrame:
    """Carga datos de múltiples experimentos y los combina en un DataFrame."""
    all_data = []
    
    for exp_path in experiment_paths:
        try:
            analysis = ExperimentAnalysis(exp_path)
            df = analysis.dataframe()
            
            # Extraer información del experimento del path
            exp_name = Path(exp_path).name
            df['experiment_name'] = exp_name
            
            # Extraer algoritmo y reward function del nombre
            parts = exp_name.split('_')
            if len(parts) >= 3:
                df['map'] = parts[0] if parts[0] in ['Spielberg', 'Catalunya', 'Monza', 'oval'] else 'unknown'
                df['algorithm'] = parts[1] if parts[1] in ['PPO', 'SAC'] else 'unknown'
                df['reward_function'] = '_'.join(parts[4:]) if len(parts) > 4 else 'unknown'
            
            all_data.append(df)
            print(f"✅ Cargado: {exp_name} ({len(df)} filas)")
            
        except Exception as e:
            print(f"❌ Error cargando {exp_path}: {e}")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\n📊 Total de datos: {len(combined_df)} filas de {len(all_data)} experimentos")
        return combined_df
    else:
        print("⚠️ No se pudieron cargar datos")
        return pd.DataFrame()

def get_experiment_paths() -> List[str]:
    """Busca automáticamente paths de experimentos."""
    base_path = Path("../models_deposito")
    
    if not base_path.exists():
        print(f"⚠️ Directorio {base_path} no existe")
        return []
    
    # Buscar directorios que parezcan experimentos
    experiment_paths = []
    for path in base_path.iterdir():
        if path.is_dir() and not path.name.startswith('.'):
            experiment_paths.append(str(path))
    
    return sorted(experiment_paths)

# Cargar datos automáticamente
experiment_paths = get_experiment_paths()
print(f"🔍 Experimentos encontrados: {len(experiment_paths)}")
for path in experiment_paths:
    print(f"  - {Path(path).name}")

# Cargar datos si hay experimentos disponibles
if experiment_paths:
    df = load_experiment_data(experiment_paths)
else:
    print("⚠️ No se encontraron experimentos. Creando DataFrame vacío para ejemplos.")
    df = pd.DataFrame()

## 📋 Resumen de Datos

In [ ]:
# Mostrar información básica del dataset
if not df.empty:
    print("📊 INFORMACIÓN DEL DATASET")
    print("=" * 50)
    print(f"Total de filas: {len(df):,}")
    print(f"Total de columnas: {len(df.columns)}")
    print(f"Experimentos únicos: {df['experiment_name'].nunique()}")
    
    # Mostrar distribución por algoritmo y reward function
    if 'algorithm' in df.columns:
        print(f"\n🤖 Distribución por Algoritmo:")
        print(df['algorithm'].value_counts())
    
    if 'reward_function' in df.columns:
        print(f"\n🎯 Distribución por Función de Recompensa:")
        print(df['reward_function'].value_counts())
    
    if 'map' in df.columns:
        print(f"\n🗺️ Distribución por Mapa:")
        print(df['map'].value_counts())
    
    # Mostrar columnas disponibles
    print(f"\n📋 Columnas Disponibles ({len(df.columns)}):")
    metrics_cols = [col for col in df.columns if any(metric in col.lower() 
                   for metric in ['reward', 'progress', 'speed', 'collision', 'time', 'duration'])]
    
    print("  📈 Métricas encontradas:")
    for col in sorted(metrics_cols):
        print(f"    - {col}")
    
    # Estadísticas básicas de métricas principales
    key_metrics = ['episode_reward_mean', 'training_iteration', 'timesteps_total']
    available_metrics = [col for col in key_metrics if col in df.columns]
    
    if available_metrics:
        print(f"\n📈 Estadísticas de Métricas Principales:")
        print(df[available_metrics].describe())
    
    # Mostrar primeras filas
    print(f"\n📄 Primeras 3 filas:")
    print(df.head(3))
    
else:
    print("⚠️ No hay datos cargados. Generando datos sintéticos para demostración...")
    
    # Crear datos sintéticos para demostración
    np.random.seed(42)
    n_samples = 1000
    
    algorithms = ['PPO', 'SAC']
    reward_functions = ['ProgressRewardAdvanced', 'SpeedReward', 'WaypointReward', 'SafetyReward']
    maps = ['Spielberg', 'Catalunya', 'Monza', 'oval_small']
    
    synthetic_data = []
    for i in range(n_samples):
        algo = np.random.choice(algorithms)
        reward_func = np.random.choice(reward_functions)
        map_name = np.random.choice(maps)
        iteration = np.random.randint(1, 201)
        
        # Generar métricas sintéticas con patrones realistas
        base_reward = np.random.normal(100, 20)
        episode_reward = base_reward + iteration * 0.5 + np.random.normal(0, 10)
        
        synthetic_data.append({
            'algorithm': algo,
            'reward_function': reward_func,
            'map': map_name,
            'training_iteration': iteration,
            'episode_reward_mean': episode_reward,
            'timesteps_total': iteration * 1000,
            'episode_len_mean': np.random.normal(200, 50),
            'custom_metrics/lap_progress': np.random.uniform(0.3, 1.0),
            'custom_metrics/avg_speed': np.random.normal(8, 2),
            'custom_metrics/episode_duration': np.random.normal(20, 5),
            'experiment_name': f"{map_name}_{algo}_Individual_Policy_{reward_func}"
        })
    
    df = pd.DataFrame(synthetic_data)
    print(f"✅ Creados {len(df)} puntos de datos sintéticos")

## 📊 1. Análisis de Rendimiento General

In [ ]:
# 1.1 Evolución del Reward por Función de Recompensa
plt.figure(figsize=(15, 8))

if 'reward_function' in df.columns and 'episode_reward_mean' in df.columns:
    # Crear subplots para cada algoritmo
    algorithms = df['algorithm'].unique() if 'algorithm' in df.columns else ['All']
    
    if len(algorithms) > 1:
        fig, axes = plt.subplots(1, len(algorithms), figsize=(15, 6))
        if len(algorithms) == 1:
            axes = [axes]
    else:
        fig, axes = plt.subplots(1, 1, figsize=(12, 6))
        axes = [axes]
    
    colors = sns.color_palette("husl", n_colors=df['reward_function'].nunique())
    
    for i, algo in enumerate(algorithms):
        ax = axes[i] if len(algorithms) > 1 else axes[0]
        
        if len(algorithms) > 1:
            algo_data = df[df['algorithm'] == algo]
        else:
            algo_data = df
        
        # Plot evolution for each reward function
        for j, reward_func in enumerate(algo_data['reward_function'].unique()):
            reward_data = algo_data[algo_data['reward_function'] == reward_func]
            
            if 'training_iteration' in reward_data.columns:
                # Group by training iteration and calculate mean
                grouped = reward_data.groupby('training_iteration')['episode_reward_mean'].agg(['mean', 'std']).reset_index()
                
                ax.plot(grouped['training_iteration'], grouped['mean'], 
                       label=reward_func, linewidth=2, color=colors[j])
                
                # Add confidence interval
                if 'std' in grouped.columns:
                    ax.fill_between(grouped['training_iteration'], 
                                   grouped['mean'] - grouped['std'], 
                                   grouped['mean'] + grouped['std'],
                                   alpha=0.2, color=colors[j])
        
        ax.set_xlabel('Iteración de Entrenamiento')
        ax.set_ylabel('Reward Promedio')
        ax.set_title(f'Evolución del Reward - {algo}' if len(algorithms) > 1 else 'Evolución del Reward')
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Columnas necesarias no encontradas para el plot de evolución de reward")

print("✅ Plot 1.1: Evolución del Reward completado")

In [ ]:
# 1.2 Matrix de Correlación de Métricas
plt.figure(figsize=(12, 10))

# Buscar columnas de métricas
metric_columns = []
potential_metrics = [
    'episode_reward_mean', 'episode_len_mean', 'timesteps_total',
    'custom_metrics/lap_progress', 'custom_metrics/avg_speed', 
    'custom_metrics/episode_duration', 'custom_metrics/lap_time',
    'info/lap_progress', 'info/avg_speed', 'info/episode_duration'
]

for col in potential_metrics:
    if col in df.columns:
        metric_columns.append(col)

# Añadir columnas adicionales que contengan métricas
for col in df.columns:
    if any(keyword in col.lower() for keyword in ['progress', 'speed', 'time', 'reward', 'collision']):
        if col not in metric_columns and df[col].dtype in ['float64', 'int64']:
            metric_columns.append(col)

if len(metric_columns) >= 2:
    # Calcular correlaciones
    correlation_data = df[metric_columns].corr()
    
    # Crear heatmap
    mask = np.triu(np.ones_like(correlation_data, dtype=bool))
    sns.heatmap(correlation_data, 
                mask=mask,
                annot=True, 
                cmap='RdBu_r', 
                center=0,
                square=True,
                fmt='.2f',
                cbar_kws={"shrink": .8})
    
    plt.title('Matrix de Correlación de Métricas')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Mostrar correlaciones más fuertes
    print("🔍 Correlaciones más fuertes (|r| > 0.5):")
    correlation_pairs = []
    for i in range(len(correlation_data.columns)):
        for j in range(i+1, len(correlation_data.columns)):
            corr_value = correlation_data.iloc[i, j]
            if abs(corr_value) > 0.5:
                correlation_pairs.append((
                    correlation_data.columns[i],
                    correlation_data.columns[j],
                    corr_value
                ))
    
    for col1, col2, corr in sorted(correlation_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {col1} ↔ {col2}: {corr:.3f}")
        
else:
    print("⚠️ No se encontraron suficientes métricas numéricas para correlación")

print("✅ Plot 1.2: Matrix de Correlación completado")

## 🏁 2. Análisis de Progreso y Tiempo de Vuelta

In [ ]:
# 2.1 Distribución de Tiempos/Duración por Reward Function
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Buscar columnas de tiempo
time_columns = []
for col in df.columns:
    if any(keyword in col.lower() for keyword in ['lap_time', 'episode_duration', 'duration']):
        if df[col].dtype in ['float64', 'int64']:
            time_columns.append(col)

if time_columns and 'reward_function' in df.columns:
    time_col = time_columns[0]  # Usar la primera columna de tiempo encontrada
    
    # Box plot de tiempos por reward function
    sns.boxplot(data=df, x='reward_function', y=time_col, ax=axes[0])
    axes[0].set_title(f'Distribución de {time_col} por Función de Recompensa')
    axes[0].set_xlabel('Función de Recompensa')
    axes[0].set_ylabel(time_col)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Violin plot para mostrar distribución completa
    sns.violinplot(data=df, x='reward_function', y=time_col, ax=axes[1])
    axes[1].set_title(f'Distribución Detallada de {time_col}')
    axes[1].set_xlabel('Función de Recompensa')
    axes[1].set_ylabel(time_col)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas por reward function
    print("📊 Estadísticas de tiempo por función de recompensa:")
    time_stats = df.groupby('reward_function')[time_col].agg(['mean', 'std', 'min', 'max']).round(2)
    print(time_stats)
    
else:
    print("⚠️ No se encontraron columnas de tiempo o reward_function")

print("✅ Plot 2.1: Distribución de Tiempos completado")

In [ ]:
# 2.2 Progreso de Vuelta vs Episodios
plt.figure(figsize=(14, 8))

# Buscar columnas de progreso
progress_columns = []
for col in df.columns:
    if any(keyword in col.lower() for keyword in ['progress', 'lap_progress']):
        if df[col].dtype in ['float64', 'int64']:
            progress_columns.append(col)

if progress_columns and 'training_iteration' in df.columns:
    progress_col = progress_columns[0]
    
    # Scatter plot con color basado en reward
    scatter = plt.scatter(df['training_iteration'], 
                         df[progress_col], 
                         c=df['episode_reward_mean'] if 'episode_reward_mean' in df.columns else 'blue',
                         alpha=0.6, 
                         cmap='viridis',
                         s=30)
    
    if 'episode_reward_mean' in df.columns:
        plt.colorbar(scatter, label='Reward Value')
    
    plt.xlabel('Iteración de Entrenamiento')
    plt.ylabel('Progreso de Vuelta')
    plt.title('Progreso de Vuelta vs Iteraciones de Entrenamiento')
    plt.grid(True, alpha=0.3)
    
    # Añadir línea de tendencia si hay suficientes datos
    if len(df) > 10:
        z = np.polyfit(df['training_iteration'], df[progress_col], 1)
        p = np.poly1d(z)
        plt.plot(df['training_iteration'], p(df['training_iteration']), "r--", alpha=0.8, linewidth=2)
    
    plt.show()
    
    # Análisis de progreso por reward function
    if 'reward_function' in df.columns:
        print("📈 Progreso promedio por función de recompensa:")
        progress_by_reward = df.groupby('reward_function')[progress_col].agg(['mean', 'std']).round(3)
        print(progress_by_reward)
    
else:
    print("⚠️ No se encontraron columnas de progreso o training_iteration")

print("✅ Plot 2.2: Progreso vs Episodios completado")

In [ ]:
# 2.3 Velocidad Promedio vs Tiempo de Vuelta (Trade-off Analysis)
plt.figure(figsize=(12, 8))

# Buscar columnas de velocidad
speed_columns = []
for col in df.columns:
    if any(keyword in col.lower() for keyword in ['speed', 'velocity']):
        if df[col].dtype in ['float64', 'int64']:
            speed_columns.append(col)

time_columns = []
for col in df.columns:
    if any(keyword in col.lower() for keyword in ['lap_time', 'episode_duration', 'duration']):
        if df[col].dtype in ['float64', 'int64']:
            time_columns.append(col)

if speed_columns and time_columns and 'reward_function' in df.columns:
    speed_col = speed_columns[0]
    time_col = time_columns[0]
    
    # Scatter plot con colores por reward function
    for i, reward_func in enumerate(df['reward_function'].unique()):
        reward_data = df[df['reward_function'] == reward_func]
        plt.scatter(reward_data[speed_col], 
                   reward_data[time_col],
                   label=reward_func,
                   alpha=0.7,
                   s=50)
    
    plt.xlabel(f'Velocidad Promedio ({speed_col})')
    plt.ylabel(f'Tiempo ({time_col})')
    plt.title('Trade-off: Velocidad vs Tiempo')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Calcular correlación velocidad-tiempo
    correlation = df[speed_col].corr(df[time_col])
    print(f"📊 Correlación {speed_col} vs {time_col}: {correlation:.3f}")
    
    # Análisis por reward function
    print("\n📈 Análisis velocidad-tiempo por función de recompensa:")
    for reward_func in df['reward_function'].unique():
        reward_data = df[df['reward_function'] == reward_func]
        avg_speed = reward_data[speed_col].mean()
        avg_time = reward_data[time_col].mean()
        print(f"  {reward_func}: Speed={avg_speed:.2f}, Time={avg_time:.2f}")
    
else:
    print("⚠️ No se encontraron columnas de velocidad, tiempo o reward_function")

print("✅ Plot 2.3: Velocidad vs Tiempo completado")

## 🚗 3. Análisis de Comportamiento de Agentes

In [ ]:
# 3.1 Comparación PPO vs SAC por Métrica (Radar Chart)
def create_radar_chart(algorithms_data, metrics, title="Comparación de Algoritmos"):
    """Crear un radar chart para comparar algoritmos."""
    from math import pi
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    
    # Número de métricas
    N = len(metrics)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]  # Cerrar el círculo
    
    colors = ['blue', 'red', 'green', 'orange']
    
    for i, (algo, data) in enumerate(algorithms_data.items()):
        values = list(data.values())
        values += values[:1]  # Cerrar el círculo
        
        ax.plot(angles, values, 'o-', linewidth=2, label=algo, color=colors[i % len(colors)])
        ax.fill(angles, values, alpha=0.25, color=colors[i % len(colors)])
    
    # Añadir labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    ax.set_ylim(0, 1)
    ax.set_title(title, size=16, y=1.1)
    ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
    ax.grid(True)
    
    return fig, ax

if 'algorithm' in df.columns and len(df['algorithm'].unique()) > 1:
    # Definir métricas para comparación
    comparison_metrics = {}
    
    # Buscar métricas disponibles
    for col in df.columns:
        if any(keyword in col.lower() for keyword in ['reward', 'progress', 'speed']):
            if df[col].dtype in ['float64', 'int64']:
                comparison_metrics[col] = col
    
    if len(comparison_metrics) >= 3:
        # Normalizar métricas (0-1) para cada algoritmo
        algorithms_data = {}
        
        for algo in df['algorithm'].unique():
            algo_data = df[df['algorithm'] == algo]
            normalized_data = {}
            
            for metric_key, metric_col in comparison_metrics.items():
                if len(algo_data[metric_col].dropna()) > 0:
                    # Normalizar usando min-max scaling global
                    global_min = df[metric_col].min()
                    global_max = df[metric_col].max()
                    if global_max != global_min:
                        normalized_value = (algo_data[metric_col].mean() - global_min) / (global_max - global_min)
                    else:
                        normalized_value = 0.5
                    normalized_data[metric_key] = max(0, min(1, normalized_value))
            
            if normalized_data:
                algorithms_data[algo] = normalized_data
        
        if len(algorithms_data) > 1:
            # Crear radar chart
            create_radar_chart(algorithms_data, list(comparison_metrics.keys()), 
                             "Comparación de Algoritmos - Métricas Normalizadas")
            plt.show()
            
            # Mostrar valores absolutos
            print("📊 Comparación de algoritmos (valores promedio):")
            comparison_df = pd.DataFrame(algorithms_data).T
            print(comparison_df.round(3))
        else:
            print("⚠️ No hay suficientes datos de algoritmos para comparar")
    else:
        print("⚠️ No se encontraron suficientes métricas para comparación")
else:
    print("⚠️ No se encontró columna 'algorithm' o solo hay un algoritmo")

print("✅ Plot 3.1: Comparación de Algoritmos completado")

In [ ]:
# 3.2 Performance por Mapa (Heatmap)
plt.figure(figsize=(14, 8))

if 'map' in df.columns and 'reward_function' in df.columns:
    # Crear pivot table para heatmap
    if 'episode_reward_mean' in df.columns:
        pivot_data = df.groupby(['map', 'reward_function'])['episode_reward_mean'].mean().reset_index()
        pivot_table = pivot_data.pivot(index='map', columns='reward_function', values='episode_reward_mean')
        
        # Crear heatmap
        sns.heatmap(pivot_table, 
                    annot=True, 
                    fmt='.1f',
                    cmap='RdYlGn',
                    cbar_kws={'label': 'Reward Promedio'})
        
        plt.title('Performance por Mapa y Función de Recompensa')
        plt.xlabel('Función de Recompensa')
        plt.ylabel('Mapa')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        
        # Análisis de dificultad por mapa
        print("🗺️ Análisis de dificultad por mapa (reward promedio):")
        map_difficulty = df.groupby('map')['episode_reward_mean'].agg(['mean', 'std', 'count']).round(2)
        map_difficulty = map_difficulty.sort_values('mean', ascending=False)
        print(map_difficulty)
        
        # Mejor combinación mapa-reward
        print("\n🏆 Mejores combinaciones mapa-reward function:")
        best_combinations = df.groupby(['map', 'reward_function'])['episode_reward_mean'].mean().nlargest(5)
        for (map_name, reward_func), score in best_combinations.items():
            print(f"  {map_name} + {reward_func}: {score:.2f}")
    
    else:
        print("⚠️ No se encontró columna 'episode_reward_mean'")
else:
    print("⚠️ No se encontraron columnas 'map' o 'reward_function'")

print("✅ Plot 3.2: Performance por Mapa completado")

## 📈 4. Análisis de Convergencia y Estabilidad

In [ ]:
# 4.1 Rolling Average del Reward y Estabilidad
fig, axes = plt.subplots(2, 1, figsize=(15, 12))

if 'training_iteration' in df.columns and 'episode_reward_mean' in df.columns:
    # Configurar ventana para rolling average
    window_size = min(50, len(df) // 10) if len(df) > 100 else 10
    
    # Plot 1: Rolling Average
    if 'reward_function' in df.columns:
        for reward_func in df['reward_function'].unique():
            reward_data = df[df['reward_function'] == reward_func].sort_values('training_iteration')
            
            if len(reward_data) > window_size:
                rolling_mean = reward_data['episode_reward_mean'].rolling(window=window_size, center=True).mean()
                rolling_std = reward_data['episode_reward_mean'].rolling(window=window_size, center=True).std()
                
                axes[0].plot(reward_data['training_iteration'], rolling_mean, 
                           label=f'{reward_func} (rolling mean)', linewidth=2)
                
                # Añadir banda de confianza
                axes[0].fill_between(reward_data['training_iteration'],
                                   rolling_mean - rolling_std,
                                   rolling_mean + rolling_std,
                                   alpha=0.2)
    
    axes[0].set_xlabel('Iteración de Entrenamiento')
    axes[0].set_ylabel('Reward Promedio (Rolling)')
    axes[0].set_title(f'Curvas de Aprendizaje Suavizadas (ventana={window_size})')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Variabilidad (Estabilidad)
    if 'reward_function' in df.columns:
        for reward_func in df['reward_function'].unique():
            reward_data = df[df['reward_function'] == reward_func].sort_values('training_iteration')
            
            if len(reward_data) > window_size:
                rolling_std = reward_data['episode_reward_mean'].rolling(window=window_size, center=True).std()
                axes[1].plot(reward_data['training_iteration'], rolling_std,
                           label=f'{reward_func}', linewidth=2)
    
    axes[1].set_xlabel('Iteración de Entrenamiento')
    axes[1].set_ylabel('Desviación Estándar (Rolling)')
    axes[1].set_title('Estabilidad del Entrenamiento')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Análisis de estabilidad
    print("📊 Análisis de estabilidad por función de recompensa:")
    if 'reward_function' in df.columns:
        stability_analysis = df.groupby('reward_function')['episode_reward_mean'].agg([
            'std', 'var', lambda x: x.std() / x.mean() if x.mean() != 0 else 0
        ]).round(3)
        stability_analysis.columns = ['std', 'variance', 'coefficient_of_variation']
        stability_analysis = stability_analysis.sort_values('coefficient_of_variation')
        print(stability_analysis)
        
        print(f"\n🏆 Función más estable: {stability_analysis.index[0]}")
        print(f"⚠️ Función menos estable: {stability_analysis.index[-1]}")

else:
    print("⚠️ No se encontraron columnas necesarias para análisis de convergencia")

print("✅ Plot 4.1: Rolling Average y Estabilidad completado")

In [ ]:
# 4.2 Tasa de Convergencia
plt.figure(figsize=(12, 8))

def calculate_convergence_metrics(data, threshold_percentile=90):
    """Calcular métricas de convergencia."""
    if len(data) < 10:
        return None, None, None
    
    # Umbral de convergencia (percentil 90 del reward final)
    final_rewards = data['episode_reward_mean'].tail(10).mean()
    threshold = final_rewards * (threshold_percentile / 100)
    
    # Encontrar cuando se alcanza el umbral por primera vez
    convergence_point = None
    for i, reward in enumerate(data['episode_reward_mean']):
        if reward >= threshold:
            convergence_point = i
            break
    
    # Calcular tasa de convergencia (pendiente promedio)
    if len(data) > 1:
        iterations = data['training_iteration'].values
        rewards = data['episode_reward_mean'].values
        convergence_rate = np.polyfit(iterations, rewards, 1)[0]  # Pendiente de regresión lineal
    else:
        convergence_rate = 0
    
    return convergence_point, threshold, convergence_rate

if 'reward_function' in df.columns and 'training_iteration' in df.columns:
    convergence_data = {}
    
    for reward_func in df['reward_function'].unique():
        reward_data = df[df['reward_function'] == reward_func].sort_values('training_iteration')
        
        if len(reward_data) > 10:
            conv_point, threshold, conv_rate = calculate_convergence_metrics(reward_data)
            convergence_data[reward_func] = {
                'convergence_episodes': conv_point,
                'convergence_rate': conv_rate,
                'final_performance': reward_data['episode_reward_mean'].tail(10).mean()
            }
    
    if convergence_data:
        # Plot de tasa de convergencia
        reward_functions = list(convergence_data.keys())
        convergence_rates = [convergence_data[rf]['convergence_rate'] for rf in reward_functions]
        final_performances = [convergence_data[rf]['final_performance'] for rf in reward_functions]
        
        colors = sns.color_palette("husl", n_colors=len(reward_functions))
        bars = plt.bar(range(len(reward_functions)), convergence_rates, color=colors)
        
        plt.xlabel('Función de Recompensa')
        plt.ylabel('Tasa de Convergencia (Reward/Iteración)')
        plt.title('Velocidad de Convergencia por Función de Recompensa')
        plt.xticks(range(len(reward_functions)), reward_functions, rotation=45, ha='right')
        plt.grid(True, alpha=0.3)
        
        # Añadir valores en las barras
        for bar, rate in zip(bars, convergence_rates):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{rate:.3f}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
        # Tabla de métricas de convergencia
        print("📊 Métricas de Convergencia:")
        convergence_df = pd.DataFrame(convergence_data).T
        convergence_df = convergence_df.round(3)
        convergence_df = convergence_df.sort_values('convergence_rate', ascending=False)
        print(convergence_df)
        
        print(f"\n🚀 Convergencia más rápida: {convergence_df.index[0]}")
        print(f"🐌 Convergencia más lenta: {convergence_df.index[-1]}")
    
    else:
        print("⚠️ No hay suficientes datos para análisis de convergencia")
else:
    print("⚠️ No se encontraron columnas necesarias para análisis de convergencia")

print("✅ Plot 4.2: Tasa de Convergencia completado")

## 🔍 5. Análisis Detallado Multi-dimensional

In [ ]:
# 5.1 Learning Curves Multi-dimensionales
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

# Definir métricas para subplot
subplot_metrics = []
for col in df.columns:
    if any(keyword in col.lower() for keyword in ['reward', 'progress', 'speed', 'duration']):
        if df[col].dtype in ['float64', 'int64'] and col != 'training_iteration':
            subplot_metrics.append(col)

# Tomar las primeras 4 métricas o rellenar con las disponibles
subplot_metrics = subplot_metrics[:4]

if len(subplot_metrics) > 0 and 'training_iteration' in df.columns:
    for i, metric in enumerate(subplot_metrics):
        if i >= 4:
            break
            
        ax = axes[i]
        
        if 'reward_function' in df.columns:
            for reward_func in df['reward_function'].unique():
                reward_data = df[df['reward_function'] == reward_func]
                
                # Agrupar por iteración y calcular promedio
                grouped = reward_data.groupby('training_iteration')[metric].mean().reset_index()
                
                if len(grouped) > 1:
                    ax.plot(grouped['training_iteration'], grouped[metric], 
                           label=reward_func, linewidth=2, marker='o', markersize=3)
        else:
            # Si no hay reward_function, plotear todos los datos
            ax.plot(df['training_iteration'], df[metric], 'b-', linewidth=2)
        
        ax.set_xlabel('Iteración de Entrenamiento')
        ax.set_ylabel(metric)
        ax.set_title(f'Evolución: {metric}')
        ax.grid(True, alpha=0.3)
        
        if 'reward_function' in df.columns and i == 0:  # Solo mostrar leyenda en el primer plot
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    # Ocultar subplots vacíos
    for i in range(len(subplot_metrics), 4):
        axes[i].set_visible(False)
    
    plt.suptitle('Evolución de Múltiples Métricas Durante el Entrenamiento', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Resumen estadístico
    print("📊 Resumen de evolución de métricas:")
    if 'reward_function' in df.columns:
        for metric in subplot_metrics:
            print(f"\n{metric}:")
            metric_summary = df.groupby('reward_function')[metric].agg(['mean', 'std', 'min', 'max']).round(3)
            print(metric_summary)
    
else:
    print("⚠️ No se encontraron métricas suficientes para plots multi-dimensionales")

print("✅ Plot 5.1: Learning Curves Multi-dimensionales completado")

In [ ]:
# 5.2 Performance vs Training Time (Eficiencia)
plt.figure(figsize=(12, 8))

if 'timesteps_total' in df.columns and 'episode_reward_mean' in df.columns:
    # Calcular tiempo de entrenamiento aproximado (usando timesteps como proxy)
    # En un setup real, podrías tener timestamps reales
    training_time_proxy = df['timesteps_total'] / 1000  # Normalizar para visualización
    
    if 'reward_function' in df.columns:
        for reward_func in df['reward_function'].unique():
            reward_data = df[df['reward_function'] == reward_func]
            
            # Obtener el performance final (último 10% de datos)
            final_data = reward_data.nlargest(int(len(reward_data) * 0.1), 'training_iteration')
            if len(final_data) > 0:
                final_performance = final_data['episode_reward_mean'].mean()
                max_training_time = reward_data['timesteps_total'].max() / 1000
                
                plt.scatter(max_training_time, final_performance, 
                           s=len(reward_data)*2,  # Tamaño basado en número de datos
                           alpha=0.7, 
                           label=reward_func)
                
                # Añadir texto con información
                plt.annotate(f'{reward_func}\n({len(reward_data)} pts)', 
                           (max_training_time, final_performance),
                           xytext=(5, 5), textcoords='offset points',
                           fontsize=8, alpha=0.8)
    
    plt.xlabel('Tiempo de Entrenamiento (timesteps / 1000)')
    plt.ylabel('Performance Final (Reward Promedio)')
    plt.title('Eficiencia del Entrenamiento: Performance vs Tiempo')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Análisis de eficiencia
    print("⚡ Análisis de eficiencia de entrenamiento:")
    if 'reward_function' in df.columns:
        efficiency_data = {}
        for reward_func in df['reward_function'].unique():
            reward_data = df[df['reward_function'] == reward_func]
            if len(reward_data) > 0:
                final_performance = reward_data['episode_reward_mean'].tail(10).mean()
                total_timesteps = reward_data['timesteps_total'].max()
                efficiency = final_performance / (total_timesteps / 1000000)  # Performance por millón de timesteps
                
                efficiency_data[reward_func] = {
                    'final_performance': final_performance,
                    'total_timesteps': total_timesteps,
                    'efficiency': efficiency
                }
        
        efficiency_df = pd.DataFrame(efficiency_data).T.round(3)
        efficiency_df = efficiency_df.sort_values('efficiency', ascending=False)
        print(efficiency_df)
        
        print(f"\n🏆 Más eficiente: {efficiency_df.index[0]}")
        print(f"⏱️ Menos eficiente: {efficiency_df.index[-1]}")

else:
    print("⚠️ No se encontraron columnas necesarias para análisis de eficiencia")

print("✅ Plot 5.2: Performance vs Training Time completado")

## 📋 6. Dashboard Integrado de Resumen

In [ ]:
# 6.1 Dashboard Completo
fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(4, 4, hspace=0.3, wspace=0.3)

if not df.empty and 'reward_function' in df.columns:
    
    # Panel 1: Evolución del Reward (2x2)
    ax1 = fig.add_subplot(gs[0:2, 0:2])
    if 'training_iteration' in df.columns and 'episode_reward_mean' in df.columns:
        for reward_func in df['reward_function'].unique():
            reward_data = df[df['reward_function'] == reward_func]
            grouped = reward_data.groupby('training_iteration')['episode_reward_mean'].mean().reset_index()
            ax1.plot(grouped['training_iteration'], grouped['episode_reward_mean'], 
                    label=reward_func, linewidth=2)
        ax1.set_title('Evolución del Reward', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Iteraciones')
        ax1.set_ylabel('Reward Promedio')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    
    # Panel 2: Distribución final de rewards
    ax2 = fig.add_subplot(gs[0, 2])
    if 'episode_reward_mean' in df.columns:
        final_rewards = df.groupby('reward_function')['episode_reward_mean'].mean()
        colors = sns.color_palette("husl", n_colors=len(final_rewards))
        bars = ax2.bar(range(len(final_rewards)), final_rewards.values, color=colors)
        ax2.set_title('Performance Final', fontsize=12, fontweight='bold')
        ax2.set_xticks(range(len(final_rewards)))
        ax2.set_xticklabels(final_rewards.index, rotation=45, ha='right', fontsize=8)
        ax2.set_ylabel('Reward')
        
        # Añadir valores en barras
        for bar, value in zip(bars, final_rewards.values):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + value*0.01,
                    f'{value:.1f}', ha='center', va='bottom', fontsize=8)
    
    # Panel 3: Mapa de calor performance por mapa
    ax3 = fig.add_subplot(gs[0, 3])
    if 'map' in df.columns and len(df['map'].unique()) > 1:
        map_performance = df.groupby(['map', 'reward_function'])['episode_reward_mean'].mean().unstack()
        if not map_performance.empty:
            sns.heatmap(map_performance, annot=True, fmt='.1f', ax=ax3, 
                       cbar_kws={'label': 'Reward'}, cmap='RdYlGn')
            ax3.set_title('Performance por Mapa', fontsize=12, fontweight='bold')
            ax3.set_xlabel('')
    else:
        ax3.text(0.5, 0.5, 'Datos de mapa\nno disponibles', 
                ha='center', va='center', transform=ax3.transAxes)
        ax3.set_title('Performance por Mapa', fontsize=12, fontweight='bold')
    
    # Panel 4: Progreso temporal
    ax4 = fig.add_subplot(gs[1, 2])
    progress_cols = [col for col in df.columns if 'progress' in col.lower()]
    if progress_cols and 'training_iteration' in df.columns:
        progress_col = progress_cols[0]
        for reward_func in df['reward_function'].unique():
            reward_data = df[df['reward_function'] == reward_func]
            grouped = reward_data.groupby('training_iteration')[progress_col].mean().reset_index()
            ax4.plot(grouped['training_iteration'], grouped[progress_col], 
                    label=reward_func, linewidth=2)
        ax4.set_title('Progreso de Vuelta', fontsize=12, fontweight='bold')
        ax4.set_xlabel('Iteraciones')
        ax4.set_ylabel('Progreso')
        ax4.grid(True, alpha=0.3)
    
    # Panel 5: Velocidad promedio
    ax5 = fig.add_subplot(gs[1, 3])
    speed_cols = [col for col in df.columns if 'speed' in col.lower()]
    if speed_cols:
        speed_col = speed_cols[0]
        speed_data = df.groupby('reward_function')[speed_col].mean()
        ax5.bar(range(len(speed_data)), speed_data.values, color=colors)
        ax5.set_title('Velocidad Promedio', fontsize=12, fontweight='bold')
        ax5.set_xticks(range(len(speed_data)))
        ax5.set_xticklabels(speed_data.index, rotation=45, ha='right', fontsize=8)
        ax5.set_ylabel('Velocidad')
    
    # Panel 6: Estabilidad (Variabilidad)
    ax6 = fig.add_subplot(gs[2, 0])
    stability_data = df.groupby('reward_function')['episode_reward_mean'].std()
    ax6.bar(range(len(stability_data)), stability_data.values, color=colors)
    ax6.set_title('Estabilidad (menos es mejor)', fontsize=12, fontweight='bold')
    ax6.set_xticks(range(len(stability_data)))
    ax6.set_xticklabels(stability_data.index, rotation=45, ha='right', fontsize=8)
    ax6.set_ylabel('Desv. Estándar')
    
    # Panel 7: Algoritmos comparison
    ax7 = fig.add_subplot(gs[2, 1])
    if 'algorithm' in df.columns and len(df['algorithm'].unique()) > 1:
        algo_performance = df.groupby(['algorithm', 'reward_function'])['episode_reward_mean'].mean().unstack()
        if not algo_performance.empty:
            algo_performance.plot(kind='bar', ax=ax7, color=colors)
            ax7.set_title('Algoritmos', fontsize=12, fontweight='bold')
            ax7.set_xlabel('Algoritmo')
            ax7.set_ylabel('Reward')
            ax7.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            ax7.tick_params(axis='x', rotation=0)
    else:
        ax7.text(0.5, 0.5, 'Comparación de\nalgoritmos no disponible', 
                ha='center', va='center', transform=ax7.transAxes)
        ax7.set_title('Algoritmos', fontsize=12, fontweight='bold')
    
    # Panel 8: Distribución de episode length
    ax8 = fig.add_subplot(gs[2, 2])
    if 'episode_len_mean' in df.columns:
        df.boxplot(column='episode_len_mean', by='reward_function', ax=ax8)
        ax8.set_title('Duración de Episodios', fontsize=12, fontweight='bold')
        ax8.set_xlabel('Reward Function')
        ax8.set_ylabel('Episode Length')
        plt.setp(ax8.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
    
    # Panel 9: Trade-off Speed vs Safety/Stability
    ax9 = fig.add_subplot(gs[2, 3])
    if speed_cols:
        safety_metric = stability_data  # Usar estabilidad como proxy de seguridad
        speed_metric = df.groupby('reward_function')[speed_cols[0]].mean()
        
        ax9.scatter(speed_metric.values, safety_metric.values, 
                   s=100, alpha=0.7, c=colors)
        
        for i, reward_func in enumerate(speed_metric.index):
            ax9.annotate(reward_func, 
                        (speed_metric.iloc[i], safety_metric.iloc[i]),
                        xytext=(5, 5), textcoords='offset points', fontsize=8)
        
        ax9.set_xlabel('Velocidad Promedio')
        ax9.set_ylabel('Variabilidad (Safety)')
        ax9.set_title('Trade-off: Velocidad vs Estabilidad', fontsize=12, fontweight='bold')
        ax9.grid(True, alpha=0.3)
    
    # Panel 10-12: Métricas adicionales en la fila inferior
    additional_metrics = []
    for col in df.columns:
        if any(keyword in col.lower() for keyword in ['collision', 'time', 'duration']) and col not in [speed_cols[0] if speed_cols else '']:
            if df[col].dtype in ['float64', 'int64']:
                additional_metrics.append(col)
    
    additional_positions = [gs[3, 0], gs[3, 1], gs[3, 2]]
    
    for i, metric in enumerate(additional_metrics[:3]):
        ax = fig.add_subplot(additional_positions[i])
        metric_data = df.groupby('reward_function')[metric].mean()
        ax.bar(range(len(metric_data)), metric_data.values, color=colors)
        ax.set_title(metric, fontsize=12, fontweight='bold')
        ax.set_xticks(range(len(metric_data)))
        ax.set_xticklabels(metric_data.index, rotation=45, ha='right', fontsize=8)
    
    # Panel resumen final
    ax_summary = fig.add_subplot(gs[3, 3])
    ax_summary.axis('off')
    
    # Crear resumen de texto
    summary_text = "📊 RESUMEN EJECUTIVO\n\n"
    
    if 'episode_reward_mean' in df.columns:
        best_reward_func = df.groupby('reward_function')['episode_reward_mean'].mean().idxmax()
        best_reward_value = df.groupby('reward_function')['episode_reward_mean'].mean().max()
        summary_text += f"🏆 Mejor Performance:\n{best_reward_func}\n({best_reward_value:.2f})\n\n"
    
    if stability_data is not None and len(stability_data) > 0:
        most_stable = stability_data.idxmin()
        summary_text += f"🎯 Más Estable:\n{most_stable}\n\n"
    
    if speed_cols and len(speed_cols) > 0:
        fastest = df.groupby('reward_function')[speed_cols[0]].mean().idxmax()
        summary_text += f"🚀 Más Rápido:\n{fastest}\n\n"
    
    summary_text += f"📈 Total Experimentos: {len(df['experiment_name'].unique()) if 'experiment_name' in df.columns else 'N/A'}\n"
    summary_text += f"📊 Total Datos: {len(df):,}"
    
    ax_summary.text(0.05, 0.95, summary_text, transform=ax_summary.transAxes,
                   fontsize=10, verticalalignment='top',
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.5))
    
    plt.suptitle('🏎️ F1TENTH Multi-Agent Training Dashboard', fontsize=20, fontweight='bold', y=0.98)
    plt.show()
    
else:
    print("⚠️ No hay suficientes datos para crear el dashboard completo")

print("✅ Dashboard Integrado completado")

## 📝 7. Conclusiones y Exportación

In [ ]:
# 7.1 Generar Conclusiones Automáticas
def generate_conclusions(df):
    """Genera conclusiones automáticas basadas en los datos."""
    conclusions = []
    
    if not df.empty and 'reward_function' in df.columns:
        
        # Análisis de performance
        if 'episode_reward_mean' in df.columns:
            performance_ranking = df.groupby('reward_function')['episode_reward_mean'].mean().sort_values(ascending=False)
            conclusions.append("🏆 RANKING DE PERFORMANCE:")
            for i, (reward_func, score) in enumerate(performance_ranking.items(), 1):
                conclusions.append(f"  {i}. {reward_func}: {score:.2f}")
        
        # Análisis de estabilidad
        if 'episode_reward_mean' in df.columns:
            stability_ranking = df.groupby('reward_function')['episode_reward_mean'].std().sort_values()
            conclusions.append("\n🎯 RANKING DE ESTABILIDAD (menor variabilidad):")
            for i, (reward_func, std) in enumerate(stability_ranking.items(), 1):
                conclusions.append(f"  {i}. {reward_func}: σ={std:.3f}")
        
        # Análisis de velocidad
        speed_cols = [col for col in df.columns if 'speed' in col.lower()]
        if speed_cols:
            speed_ranking = df.groupby('reward_function')[speed_cols[0]].mean().sort_values(ascending=False)
            conclusions.append(f"\n🚀 RANKING DE VELOCIDAD ({speed_cols[0]}):")
            for i, (reward_func, speed) in enumerate(speed_ranking.items(), 1):
                conclusions.append(f"  {i}. {reward_func}: {speed:.2f}")
        
        # Análisis de eficiencia
        if 'timesteps_total' in df.columns and 'episode_reward_mean' in df.columns:
            efficiency_data = {}
            for reward_func in df['reward_function'].unique():
                reward_data = df[df['reward_function'] == reward_func]
                if len(reward_data) > 0:
                    final_performance = reward_data['episode_reward_mean'].tail(10).mean()
                    total_timesteps = reward_data['timesteps_total'].max()
                    efficiency = final_performance / (total_timesteps / 1000000)
                    efficiency_data[reward_func] = efficiency
            
            if efficiency_data:
                efficiency_ranking = sorted(efficiency_data.items(), key=lambda x: x[1], reverse=True)
                conclusions.append("\n⚡ RANKING DE EFICIENCIA (performance/tiempo):")
                for i, (reward_func, eff) in enumerate(efficiency_ranking, 1):
                    conclusions.append(f"  {i}. {reward_func}: {eff:.3f}")
        
        # Recomendaciones
        conclusions.append("\n💡 RECOMENDACIONES:")
        
        if 'episode_reward_mean' in df.columns:
            best_overall = df.groupby('reward_function')['episode_reward_mean'].mean().idxmax()
            conclusions.append(f"  • Para máximo performance: {best_overall}")
        
        if 'episode_reward_mean' in df.columns:
            most_stable = df.groupby('reward_function')['episode_reward_mean'].std().idxmin()
            conclusions.append(f"  • Para mayor estabilidad: {most_stable}")
        
        # Análisis de mapas
        if 'map' in df.columns and len(df['map'].unique()) > 1:
            map_difficulty = df.groupby('map')['episode_reward_mean'].mean().sort_values(ascending=False)
            easiest_map = map_difficulty.index[0]
            hardest_map = map_difficulty.index[-1]
            conclusions.append(f"  • Mapa más fácil: {easiest_map}")
            conclusions.append(f"  • Mapa más desafiante: {hardest_map}")
        
        # Estadísticas generales
        conclusions.append(f"\n📊 ESTADÍSTICAS GENERALES:")
        conclusions.append(f"  • Total de experimentos: {len(df['experiment_name'].unique()) if 'experiment_name' in df.columns else 'N/A'}")
        conclusions.append(f"  • Total de datos recolectados: {len(df):,}")
        conclusions.append(f"  • Funciones de recompensa evaluadas: {len(df['reward_function'].unique())}")
        
        if 'algorithm' in df.columns:
            conclusions.append(f"  • Algoritmos comparados: {', '.join(df['algorithm'].unique())}")
        
        if 'map' in df.columns:
            conclusions.append(f"  • Mapas evaluados: {', '.join(df['map'].unique())}")
    
    return conclusions

# Generar y mostrar conclusiones
print("🎯 ANÁLISIS AUTOMÁTICO DE RESULTADOS")
print("=" * 60)

conclusions = generate_conclusions(df)
for conclusion in conclusions:
    print(conclusion)

print("\n" + "=" * 60)

In [ ]:
# 7.2 Exportar Datos y Resultados
import json
from datetime import datetime

def export_analysis_results(df, conclusions):
    """Exporta los resultados del análisis a diferentes formatos."""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    export_dir = Path("analysis_exports")
    export_dir.mkdir(exist_ok=True)
    
    exported_files = []
    
    # 1. Exportar DataFrame completo a CSV
    if not df.empty:
        csv_path = export_dir / f"training_data_{timestamp}.csv"
        df.to_csv(csv_path, index=False)
        exported_files.append(str(csv_path))
        print(f"✅ Datos exportados a: {csv_path}")
    
    # 2. Exportar resumen estadístico
    if not df.empty and 'reward_function' in df.columns:
        summary_data = {}
        
        # Métricas por reward function
        for reward_func in df['reward_function'].unique():
            reward_data = df[df['reward_function'] == reward_func]
            
            metrics = {}
            for col in df.columns:
                if col in ['episode_reward_mean', 'custom_metrics/lap_progress', 'custom_metrics/avg_speed']:
                    if col in reward_data.columns and reward_data[col].dtype in ['float64', 'int64']:
                        metrics[col] = {
                            'mean': float(reward_data[col].mean()),
                            'std': float(reward_data[col].std()),
                            'min': float(reward_data[col].min()),
                            'max': float(reward_data[col].max()),
                            'count': int(reward_data[col].count())
                        }
            
            summary_data[reward_func] = metrics
        
        # Exportar como JSON
        json_path = export_dir / f"summary_statistics_{timestamp}.json"
        with open(json_path, 'w') as f:
            json.dump(summary_data, f, indent=2)
        exported_files.append(str(json_path))
        print(f"✅ Estadísticas exportadas a: {json_path}")
    
    # 3. Exportar conclusiones
    conclusions_path = export_dir / f"conclusions_{timestamp}.txt"
    with open(conclusions_path, 'w') as f:
        f.write("F1TENTH MULTI-AGENT TRAINING ANALYSIS\\n")
        f.write("=" * 50 + "\\n")
        f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\\n\\n")
        for conclusion in conclusions:
            f.write(conclusion + "\\n")
    exported_files.append(str(conclusions_path))
    print(f"✅ Conclusiones exportadas a: {conclusions_path}")
    
    # 4. Crear reporte HTML simple
    html_path = export_dir / f"report_{timestamp}.html"
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>F1TENTH Training Analysis Report</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 40px; }}
            h1 {{ color: #333; }}
            .metric {{ background: #f5f5f5; padding: 10px; margin: 10px 0; border-radius: 5px; }}
            .conclusion {{ background: #e8f4fd; padding: 15px; margin: 10px 0; border-radius: 5px; }}
        </style>
    </head>
    <body>
        <h1>🏎️ F1TENTH Multi-Agent Training Analysis</h1>
        <p><strong>Generated:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        
        <h2>📊 Data Summary</h2>
        <div class="metric">
            <strong>Total Records:</strong> {len(df):,}<br>
            <strong>Reward Functions:</strong> {len(df['reward_function'].unique()) if 'reward_function' in df.columns else 'N/A'}<br>
            <strong>Experiments:</strong> {len(df['experiment_name'].unique()) if 'experiment_name' in df.columns else 'N/A'}
        </div>
        
        <h2>🎯 Key Conclusions</h2>
        <div class="conclusion">
            {"<br>".join(conclusions)}
        </div>
        
        <h2>📁 Exported Files</h2>
        <ul>
            {"".join([f"<li>{file}</li>" for file in exported_files])}
        </ul>
    </body>
    </html>
    """
    
    with open(html_path, 'w') as f:
        f.write(html_content)
    exported_files.append(str(html_path))
    print(f"✅ Reporte HTML exportado a: {html_path}")
    
    return exported_files

# Ejecutar exportación
print("📁 EXPORTANDO RESULTADOS DEL ANÁLISIS...")
print("-" * 40)

exported_files = export_analysis_results(df, conclusions)

print(f"\\n🎉 Exportación completada!")
print(f"📁 {len(exported_files)} archivos creados en 'analysis_exports/'")
print("\\nArchivos generados:")
for file in exported_files:
    print(f"  📄 {file}")

print("\\n💡 Tip: Usa estos archivos para:")
print("  • Compartir resultados con el equipo")
print("  • Crear presentaciones")
print("  • Documentar experimentos")
print("  • Comparar con entrenamientos futuros")

## 🚀 Próximos Pasos y Uso del Notebook

### Cómo usar este notebook:

1. **Primera ejecución**: Ejecuta todas las celdas para ver el análisis con datos sintéticos
2. **Con datos reales**: Modifica la función `get_experiment_paths()` para apuntar a tus experimentos
3. **Personalización**: Ajusta los plots según tus métricas específicas
4. **Automatización**: Ejecuta el notebook después de cada entrenamiento

### Funcionalidades incluidas:

✅ **Análisis de Performance**: Evolución de rewards, comparación entre funciones  
✅ **Análisis de Convergencia**: Estabilidad, tasa de convergencia  
✅ **Comparación de Algoritmos**: PPO vs SAC con radar charts  
✅ **Análisis por Mapa**: Performance en diferentes circuitos  
✅ **Dashboard Integrado**: Vista general de todos los resultados  
✅ **Exportación Automática**: CSV, JSON, HTML y conclusiones  
✅ **Conclusiones Automáticas**: Análisis y recomendaciones generadas  

### Para extender el análisis:

- Añade nuevas métricas en la sección de `subplot_metrics`
- Modifica los callbacks en `lib/callbacks.py` para recolectar más datos
- Personaliza los colores y estilos en la configuración inicial
- Añade análisis específicos para tu caso de uso

### Integración con experimentos:

Este notebook está diseñado para trabajar con tu configuración actual:
- Lee datos de `../models_deposito/`
- Compatible con `experiments_victor.yaml`
- Analiza métricas de `callbacks.py`
- Funciona con todas las reward functions en `rewards.py`

¡Ahora tienes un sistema completo de análisis para documentar y optimizar tus entrenamientos! 🎯